# Live spatial analysis: reactive human lymph node
### WCI 2026 - Single cell/spatial transcriptomics course
### CSBL - PhD student Guilherme de-Mira

> ⚠️ **Instructor note — each student opens their own fresh Colab session from the QR code, so "run it ahead of time" only applies to your own demo copy, not theirs.** Build the timing into the class instead:
>
> 1. **You**: run this entire notebook once, start to finish, before class — so you have a fully executed reference on your own screen no matter what happens live.
> 2. **When you reveal the QR code to the room**: have everyone scan it, then immediately have them run only the install cell and the data-download cell (~1–1.5 min combined) — while that runs on ~30 laptops in parallel, keep talking (Section 1's dataset background is exactly the right length of content to cover here).
> 3. Keep the **BACKUP_EXECUTED** copy of this notebook open on your own laptop as a fallback you can screen-share if the venue wifi struggles under everyone downloading at once.

In [ ]:
# We are going to install the packages needed for this notebook.
!pip install -q scanpy squidpy igraph

In [ ]:
# Now, we import the packages we will use in this session.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
from matplotlib.collections import PolyCollection
from matplotlib.lines import Line2D

sc.settings.verbosity = 0
%matplotlib inline

## 1. The data: a real Xenium 5K human lymph node

This is a public dataset from 10x Genomics: a **reactive human lymph node**, profiled with the **Xenium Human 5K panel** (4,624 genes measured in situ, at single-cell resolution). It also carries each cell's real segmented outline, not just its center point — we'll use that below to see the cells as they actually look in the tissue.

In [ ]:
# Here, we download the dataset used in this notebook.
import os

DATA_URL = "https://github.com/guilhermemira266/xenium-lymph-node-demo/releases/download/v1/xenium_lymph_node_5k_crop.h5ad"
fname = "xenium_lymph_node_5k_crop.h5ad"

if not os.path.exists(fname):
    !wget -q "{DATA_URL}" -O "{fname}"

In [ ]:
# Now we use a function from the scanpy package to read the dataset (h5ad format).
adata = sc.read_h5ad(fname)
# We print it below to check that everything loaded correctly.
adata

## 2. Quality control

In [ ]:
# Here, we filter out cells with less than 10 transcript reads, and genes
# detected in fewer than 3 cells.
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
# A quick visual check: after filtering, how many transcripts does a
# typical cell have? No cell should be sitting at (or near) zero.
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(adata.obs["total_counts"], bins=40, color="#4A90E2")
ax.set_xlabel("total transcripts per cell")
ax.set_ylabel("number of cells")
ax.set_title("Transcript counts after filtering")
plt.tight_layout()

In [ ]:
# Before we perform normalization and log-transformation, we save the
# raw counts in a separate layer.
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

# We print it again to check that everything looks correct.
adata

## 3. Cluster cells by their expression profile

> ⏱️ **This cell takes ~30–40 seconds to run** — that's expected, not a crash. It's a one-time "warm-up" cost from a library compiling code the first time it's called in a fresh session, not about the size of the data.

- **PCA**: compresses thousands of genes into a handful of numbers that capture most of the differences between cells.
- **Nearest neighbors**: connects each cell to the other cells most similar to it.
- **Leiden clustering**: groups cells that are densely connected into clusters.

In [ ]:
sc.pp.pca(adata, n_comps=30, random_state=0)
sc.pp.neighbors(adata, random_state=0)

# resolution: higher values give more, smaller clusters
# flavor="igraph": a faster clustering implementation (uses the igraph package)
# n_iterations: how many refinement passes the algorithm runs
# random_state: fixes the randomness so we all get the same clusters
sc.tl.leiden(adata, resolution=1.0, flavor="igraph", n_iterations=2, random_state=0)

# Here, we check the number of cells in each cluster.
adata.obs["leiden"].value_counts()

## Spatial plot of the data using cell shapes/boundaries

Every cell in this dataset also carries its actual segmented outline (a polygon). Let's build those shapes once, and reuse them for every spatial plot below.

In [ ]:
boundaries = adata.uns["cell_boundaries"]

# group the boundary points by cell, so each cell has its own list of (x, y) corners
cell_polygons = {
    cid: g[["vertex_x", "vertex_y"]].values
    for cid, g in boundaries.groupby("cell_id", sort=False)
}
print(f"{len(cell_polygons):,} real cell shapes loaded")

BG_COLOR = "#0b0d12"
FILTERED_COLOR = "#20242c"  # cells dropped by QC, or without a label yet


def plot_by_boundary(color_col, palette, title, figsize=(7, 7)):
    # pick one color per cell, based on its label
    labels = adata.obs[color_col]
    colors = [
        palette.get(labels[cid], FILTERED_COLOR) if cid in labels.index else FILTERED_COLOR
        for cid in cell_polygons
    ]
    # draw every cell shape at once, each filled with its color
    fig, ax = plt.subplots(figsize=figsize, facecolor=BG_COLOR)
    ax.add_collection(PolyCollection(
        list(cell_polygons.values()), facecolors=colors,
        edgecolors="black", linewidths=0.15, alpha=0.85,
    ))
    ax.set_facecolor(BG_COLOR)
    ax.set_xlim(boundaries["vertex_x"].min(), boundaries["vertex_x"].max())
    ax.set_ylim(boundaries["vertex_y"].max(), boundaries["vertex_y"].min())
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title, color="white")

    # build the color legend
    handles = [
        Line2D([0], [0], marker="s", linestyle="none", markersize=10,
               markerfacecolor=c, markeredgecolor="none", label=k)
        for k, c in palette.items()
    ]
    handles.append(Line2D([0], [0], marker="s", linestyle="none", markersize=10,
                           markerfacecolor=FILTERED_COLOR, markeredgecolor="none", label="filtered / unlabeled"))
    ax.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0),
              frameon=False, labelcolor="white", fontsize=9)
    return fig, ax

## 4. Spatial plot of the clusters

How are the clusters spatially distributed across the tissue?

In [ ]:
leiden_cats = adata.obs["leiden"].cat.categories
leiden_palette = {cat: plt.cm.tab10(i % 10) for i, cat in enumerate(leiden_cats)}

plot_by_boundary("leiden", leiden_palette, "leiden clusters — real cell shapes")

## 5. What genes define each cluster?

For each cluster, we are going to find the genes that are most specifically high in that cluster versus all others.

In [ ]:
# We use the Wilcoxon rank-sum test because it doesn't assume the data
# follows a normal distribution — a safer choice for sparse, non-normal
# single-cell count data like this.
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")

# For each cluster, grab its top 5 marker genes and print them next to the cluster size.
for cl in adata.obs["leiden"].cat.categories:
    top_genes = adata.uns["rank_genes_groups"]["names"][cl][:5]
    print(f"cluster {cl:>2} (n={ (adata.obs['leiden']==cl).sum() :>4}):  {', '.join(top_genes)}")

**Walk through 2–3 of these live** — you don't need to read every row out loud:
- One cluster's top genes will be `CD3E`, `CD2`, `TCF7`... → unmistakably **T cells**.
- Another will show `MS4A1`, `CD79A`, `CD19`... → **B cells**.
- Point out one "wow" rare population if you have time — e.g. a cluster marked by `CR2`, `CXCL13`, `VCAM1` is a **follicular dendritic cell** (a stromal cell, not even a classic immune cell, that organizes B cell follicles), or one marked by `CLEC4C`, `IL3RA`, `GZMB` is a **plasmacytoid dendritic cell** — a rare interferon-producing population most people never expect to see pop out of an unsupervised clustering run in a few seconds.

## 6. Naming the clusters

Rather than hand-typing "cluster 4 = B cells" (the exact cluster *numbers* aren't guaranteed to come out the same on every machine — different package versions can shuffle them), we score every cell against a short marker list per candidate cell type, average each score within each cluster, and give each cluster the label it scores highest on. Same "winner-takes-all" logic behind the naming, just automated instead of hardcoded.

In [ ]:
marker_sets = {
    "T cells": ["CD3E", "CD2"],
    "B cells": ["MS4A1", "CD79A", "CD19"],
    "Dendritic cells": ["FSCN1", "LAMP3"],
    "pDCs": ["CLEC4C", "IL3RA", "GZMB"],
    "Macrophages": ["SLC40A1", "MMP9"],
    "Follicular dendritic cells": ["CR2", "CXCL13"],
    "Fibroblastic reticular cells": ["CCL19", "CXCL12"],
    "Endothelial cells": ["PECAM1", "PLVAP"],
}
for name, genes in marker_sets.items():
    sc.tl.score_genes(adata, genes, score_name=f"score_{name}")

score_cols = [f"score_{name}" for name in marker_sets]
cluster_scores = adata.obs.groupby("leiden", observed=True)[score_cols].mean()
cluster_scores.columns = list(marker_sets.keys())
best_label_per_cluster = cluster_scores.idxmax(axis=1)
print(best_label_per_cluster)

adata.obs["cell_type"] = adata.obs["leiden"].map(best_label_per_cluster).astype("category")
adata.obs["cell_type"].value_counts()

In [ ]:
cell_type_palette = {
    "T cells": "#2ECC71",
    "B cells": "#4A90E2",
    "Dendritic cells": "#F1C40F",
    "pDCs": "#E67E22",
    "Macrophages": "#E84393",
    "Follicular dendritic cells": "#9B59B6",
    "Fibroblastic reticular cells": "#95A5A6",
    "Endothelial cells": "#FF3B30",
}

plot_by_boundary("cell_type", cell_type_palette, "cell_type — real cell shapes")

**This is the reveal moment.** Point at the plot: on one side, a dense field of B cells (a *lymphoid follicle* / germinal center) with a handful of follicular dendritic cells and macrophages embedded in it; on the other side, T cells threaded through by fibroblastic reticular cells and blood vessels (a *paracortex* / T cell zone). This is the classic microanatomy of a lymph node — and no one told the algorithm any of it.

## 7. Spatial niches: grouping by microenvironment, not by cell identity

So far we grouped cells by what they *are* (expression → cell type). Now let's group them by *where they sit*: for every cell, look at its 20 nearest spatial neighbors and ask "what mix of cell types surrounds this cell?" Cells with a similar surrounding mix — even if they are different cell types themselves — get grouped into the same **niche**.

In [ ]:
from sklearn.cluster import KMeans

N_NEIGHBORS = 20
N_NICHES = 6

sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=N_NEIGHBORS)
adjacency = adata.obsp["spatial_connectivities"].copy()
adjacency.setdiag(1)  # a cell's own type also counts as part of its neighborhood

cell_types = adata.obs["cell_type"].cat.categories.tolist()
onehot = pd.get_dummies(adata.obs["cell_type"]).reindex(columns=cell_types, fill_value=0).values.astype(float)
neighbor_counts = adjacency @ onehot
composition = neighbor_counts / neighbor_counts.sum(axis=1, keepdims=True)

kmeans = KMeans(n_clusters=N_NICHES, random_state=0, n_init=10)
adata.obs["niche"] = pd.Categorical([f"Niche {i}" for i in kmeans.fit_predict(composition)])

adata.obs["niche"].value_counts()

## 8. What is each niche actually made of?

Same idea as the marker-gene table earlier, but for composition instead of genes: average the neighborhood mix within each niche.

In [ ]:
comp_df = pd.DataFrame(composition, columns=cell_types, index=adata.obs_names)
comp_df["niche"] = adata.obs["niche"].values
niche_profile = comp_df.groupby("niche", observed=True)[cell_types].mean()

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(niche_profile.values, aspect="auto", cmap="viridis")
ax.set_xticks(range(len(cell_types)))
ax.set_xticklabels(cell_types, rotation=45, ha="right")
ax.set_yticks(range(len(niche_profile)))
ax.set_yticklabels(niche_profile.index)
plt.colorbar(im, ax=ax, label="mean fraction of neighborhood")
ax.set_title(f"Niche composition ({N_NICHES} niches, {N_NEIGHBORS} spatial neighbors)")
plt.tight_layout()

**Keep this heatmap visible** — you'll need it in a moment to tell which niche number is which, since *which* niche gets called "Niche 0" vs "Niche 3" can vary run to run. Read each row: whichever cell type has the brightest cell in that row is what dominates that niche.

Now let's see where these niches actually sit in the tissue.

In [ ]:
niche_cats = sorted(adata.obs["niche"].cat.categories, key=lambda x: int(x.split()[1]))
niche_palette = {cat: plt.cm.tab10(i % 10) for i, cat in enumerate(niche_cats)}

plot_by_boundary("niche", niche_palette, "Spatial niches — real cell shapes")

**Point out the shapes, not just the colors**: a niche dominated by B cells should form a solid core; a mixed B+T niche often forms a *ring* around it (the mantle zone wrapping the germinal center); a niche enriched for endothelial cells tends to trace thin diagonal streaks (a vessel or trabecula running through the tissue); a stromal/fibroblastic-reticular-cell niche often hugs the outer edge of the section (the capsule). None of this was told to the algorithm — it fell out of nothing but "what surrounds each cell."

## 9. From discrete niches to a continuous question

Instead of splitting cells into two niche groups, let's ask a more direct question: does a B cell's state change smoothly with how far it is from the nearest T cell? We measure the real physical distance for every B cell, then check which genes track that distance.

In [ ]:
from scipy.spatial import cKDTree
from scipy.stats import spearmanr, false_discovery_control

# for every B cell, find the distance to the nearest T cell
b_mask = (adata.obs["cell_type"] == "B cells").values
t_mask = (adata.obs["cell_type"] == "T cells").values
tree = cKDTree(adata.obsm["spatial"][t_mask])
distance_to_T, _ = tree.query(adata.obsm["spatial"][b_mask], k=1)

b_adata = adata[b_mask].copy()
b_adata.obs["distance_to_nearest_T_cell"] = distance_to_T
b_adata.obs[["distance_to_nearest_T_cell"]].describe()

## 10. Which genes track that distance?

For every gene, we test whether its expression correlates with distance to the nearest T cell — using a correlation that only cares about rank order (Spearman), not exact values, since single-cell counts are noisy.

In [ ]:
X = b_adata.X.toarray()

rhos, pvals, genes_tested = [], [], []
for j, gene in enumerate(adata.var_names):
    column = X[:, j]
    if column.std() == 0:
        continue
    rho, p = spearmanr(column, distance_to_T)
    rhos.append(rho)
    pvals.append(p)
    genes_tested.append(gene)

corr = pd.DataFrame({"gene": genes_tested, "rho": rhos, "pval": pvals})
corr["padj"] = false_discovery_control(corr["pval"])
sig = corr[corr["padj"] < 0.05]
print(f"{len(sig)} / {len(corr)} genes significantly correlated with distance")

sig.sort_values("rho", ascending=False).head(10)[["gene", "rho", "padj"]]

**Genes at the top — higher the *farther* a B cell is from any T cell — include `BCL6`, `LMO2` and `EBI3`, all textbook markers of the germinal center reaction** (the process that builds high-affinity antibodies). Genes at the *bottom* of the same list tend to be T cell identity genes (`CD3E`, `CD247`...) — B cells right at the edge of the T zone picking up a faint trace of their neighbors' signal, the kind of effect you'd expect in any densely packed tissue.

## 11. Visualizing the trend

Individual cells are noisy (most genes here are detected in only a handful of transcripts per cell), so instead of a raw scatterplot, we group B cells into distance bins and plot the *percentage of cells expressing each gene* per bin.

In [ ]:
genes_to_plot = ["BCL6", "LMO2", "EBI3"]
n_bins = 6
bin_id = pd.qcut(distance_to_T, n_bins, labels=False)
bin_edges = pd.qcut(distance_to_T, n_bins).categories

fig, ax = plt.subplots(figsize=(8, 5))
for gene, color in zip(genes_to_plot, ["#9B59B6", "#3498DB", "#E67E22"]):
    values = np.asarray(b_adata[:, gene].layers["counts"].todense()).flatten()
    pct_positive = [(values[bin_id == i] > 0).mean() * 100 for i in range(n_bins)]
    ax.plot(range(n_bins), pct_positive, marker="o", linewidth=2, label=gene, color=color)

ax.set_xticks(range(n_bins))
ax.set_xticklabels([f"{iv.left:.0f}–{iv.right:.0f}" for iv in bin_edges])
ax.set_xlabel("distance to nearest T cell (µm), binned")
ax.set_ylabel("% of B cells expressing the gene")
ax.set_title("Germinal center genes vs. distance from the T cell zone")
ax.legend()
plt.tight_layout()

## 🔎 The insight

As B cells sit farther from the T cell zone — deeper into the follicle — a bigger fraction of them express `BCL6`, `LMO2` and `EBI3`: three independent, well-established markers of the **germinal center reaction**. None of these three genes is a T cell gene, and none of them was used anywhere in this notebook to define cell types — this pattern fell out purely from combining real spatial position with expression.

**Same idea as grouping cells into niches, just told with a continuous measurement instead of discrete groups**: physical location inside a lymphoid organ isn't just geography — distance from the T cell zone lines up with how far a B cell has committed to the germinal center program.

## (Bonus — only if time allows)

- Re-run the clustering at `resolution=0.5` or `resolution=1.5` and see which of today's clusters merge or split.
- Try the mirror version: anchor on **T cells** instead, and compute distance to the nearest **B cell** — which T cell genes track that distance?
- Look at one of the `score_*` columns computed earlier directly, per cell instead of averaged per cluster: `sq.pl.spatial_scatter(adata, color="score_B cells", shape=None, size=6)` — a continuous alternative to discrete cluster labels.